In [28]:
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
import yaml

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.block(x)

class BigGenerator(nn.Module):
    """
    B = Batch size
    J = Destination zones
    K = Employment Categories

    Inputs:
        - Vector:
            - skim data (probably time) (B, J)
            - Employment data (B, J * K)
        - Hidden dim
        - Dropout probability
    """
    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        hidden_dim: int = 128,
        dropout: float = 0.1,
        depth: int = 3,
    ):
        super().__init__()
        self.num_outcomes = output_dim
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.inp = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.Sequential(*[ResBlock(hidden_dim, dropout) for _ in range(depth)])
        self.out = nn.Sequential(nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, output_dim))
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)

    def forward(self, data_vect: torch.Tensor) -> torch.Tensor:
        n = self.inp(data_vect)
        n = self.blocks(n)
        return self.out(n)
    
def load_model(model_path, generator_class, model_config):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    generator = generator_class(**model_config).to(device)
    if isinstance(checkpoint, dict):
        if 'G_state_dict' in checkpoint:
            generator.load_state_dict(checkpoint['G_state_dict'])
        elif 'model_state_dict' in checkpoint:
            generator.load_state_dict(checkpoint['model_state_dict'])
        else:
            generator.load_state_dict(checkpoint)
    else:
        generator.load_state_dict(checkpoint)
    generator.eval()
    return generator, device

gan_model_file = r"C:\models\Reno_TDM\scenarios\base_2022_gan\input\resident\dc\GAN\N_HBSHP\best_model.pt"
param_file = r"C:\models\Reno_TDM\scenarios\base_2022_gan\input\resident\dc\GAN\N_HBSHP\model_setup.yaml"


param_array_in = yaml.load(open(param_file), Loader = yaml.SafeLoader)
param_array = {
    "output_dim": param_array_in['output_dim'],
    "input_dim": param_array_in['input_dim'],
    "hidden_dim": param_array_in['hidden'],
    "dropout": param_array_in['dropout'],
    "depth": param_array_in['depth'],
    }
G, device = load_model(
            gan_model_file,
            BigGenerator,
            param_array
        )

ZONES = 1164 


In [29]:
G.eval()
z = torch.ones([39607])
z = z.clone().detach().requires_grad_(True)
y = G(z)
scalar = y[1148]

G.zero_grad()
scalar.backward()
saliency = z.grad.abs()
print(saliency)

tensor([7.2310e-03, 1.0141e-02, 7.6692e-03,  ..., 7.7981e-05, 4.9183e-04,
        1.1326e-03])


In [30]:
# The output...
# Cost Matrix (1164)
# Origin Chars
# Destination Chars
# Intradistrict (if)
# Intrazonal (if)

In [31]:
skim_saliency = saliency[:1164].numpy()

dest_dv_start = ZONES + len(param_array_in['ose_cols'])

saliency_table = pd.DataFrame({'cost': skim_saliency})
i_zone = 0
for l in param_array_in['dse_cols']:
    saliency_table[l] = saliency[dest_dv_start + i_zone * ZONES: dest_dv_start + (1 + i_zone) * ZONES].numpy()
    i_zone += 1

saliency_table.to_clipboard()



In [ ]:
ose_dist_saliency = saliency[1165: 1165 + len(param_array_in['ose_cols'])].numpy()
pd.DataFrame({'ose_cols': ose_dist_saliency}).to_clipboard()

array([2.5922975e-03, 4.9559783e-02, 1.0155177e-01, 5.9440296e-02,
       1.6419448e-02, 9.1235237e-03, 6.9098294e-02, 7.4015401e-02,
       2.7760779e-02, 5.9155498e-02, 5.1294238e-04, 1.0956907e-04,
       1.7328074e-04, 3.1820557e-04, 1.7646824e-04, 6.4963597e-04,
       1.7719667e-05, 8.6352497e-04, 4.4174076e-04, 1.0597290e-03,
       3.5224998e-04, 1.6412822e-03, 1.1329490e-01, 1.2843446e-01,
       5.9244681e-02, 9.6549123e-04, 7.8935485e-04, 1.5445547e-04,
       1.3978552e-04, 8.9628925e-04, 9.4850804e-04], dtype=float32)

: 